# Data Mining Assignment — Apache Spark Data Lake Preprocessing

**Dataset:** `sales.csv`  
**Environment:** Google Colab + PySpark

## Objective
Create a simple Data Lake structure with:
- **Bronze:** original `sales.csv`, kept unchanged
- **Silver:** basic preprocessing using Apache Spark

The notebook inspects the data, performs basic preprocessing, saves the Silver output, and verifies the result.

In [52]:

!pip install -q pyspark

In [53]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    initcap,
    lower,
    when,
    coalesce,
    to_date,
    sum as spark_sum
)
import os
import shutil
import glob

spark = SparkSession.builder     .appName("Data Mining Spark Data Lake Assignment")     .getOrCreate()

print("SparkSession created successfully!")
print("Spark version:", spark.version)

SparkSession created successfully!
Spark version: 4.0.4


## 1. Create the Data Lake structure

The Bronze layer will contain an unchanged copy of the original dataset.  
The Silver layer will contain the preprocessed Spark output.

In [54]:


os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver/preprocessed_sales", exist_ok=True)

print("Data Lake structure created:")
print("/content/data_lake/bronze/")
print("/content/data_lake/silver/preprocessed_sales/")

Data Lake structure created:
/content/data_lake/bronze/
/content/data_lake/silver/preprocessed_sales/


In [55]:


possible_paths = [
    "/content/sample_data/sales.csv",
    "/content/sales.csv"
]

source_file = next((p for p in possible_paths if os.path.exists(p)), None)

if source_file is None:
    raise FileNotFoundError(
        "sales.csv was not found. Upload sales.csv to Colab and run this cell again."
    )

print("Source dataset found at:", source_file)

Source dataset found at: /content/sample_data/sales.csv


In [56]:


bronze_file = "/content/data_lake/bronze/sales.csv"


shutil.copy2(source_file, bronze_file)

print("Original sales.csv copied to Bronze layer.")
print("Bronze file:", bronze_file)
print("Bronze file exists:", os.path.exists(bronze_file))

Original sales.csv copied to Bronze layer.
Bronze file: /content/data_lake/bronze/sales.csv
Bronze file exists: True


## 2. Load and inspect the Bronze layer

The Bronze file is read with Spark. The original Bronze CSV is never modified.

In [57]:


bronze_path = "/content/data_lake/bronze/sales.csv"

df = spark.read     .option("header", True)     .option("inferSchema", True)     .csv(bronze_path)

print("Bronze data loaded successfully.")

Bronze data loaded successfully.


In [58]:


print("SCHEMA")
print("=" * 60)
df.printSchema()

print("\nCOLUMNS")
print("=" * 60)
print(df.columns)

print("\nSAMPLE ROWS")
print("=" * 60)
df.show(10, truncate=False)

initial_count = df.count()

print("\nTOTAL RECORD COUNT")
print("=" * 60)
print("Records before preprocessing:", initial_count)

SCHEMA
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)


COLUMNS
['order_id', 'customer_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'discount_percent', 'payment_method', 'city', 'state', 'order_date', 'order_status']

SAMPLE ROWS
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |categ

## 3. Data quality inspection

The following Spark operations identify:
- exact duplicate rows
- missing values
- text formatting inconsistencies
- invalid quantity, unit price, and discount values
- inconsistent or invalid dates

In [59]:


deduplicated_count = df.dropDuplicates().count()
duplicate_count = initial_count - deduplicated_count

print("Total records:", initial_count)
print("Exact duplicate rows:", duplicate_count)

print("\nDuplicate records (if any):")
df.groupBy(df.columns)     .count()     .filter(col("count") > 1)     .show(truncate=False)

Total records: 1000
Exact duplicate rows: 10

Duplicate records (if any):
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|10029

In [60]:


null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

print("NULL values by column:")
null_counts.show(truncate=False)

NULL values by column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [61]:


text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

print("Distinct text values:")
for c in text_columns:
    print("\n" + "=" * 60)
    print("COLUMN:", c)
    print("=" * 60)
    df.select(c).distinct().orderBy(c).show(100, truncate=False)

print("\nLeading/trailing whitespace counts:")
for c in text_columns:
    count = df.filter(
        col(c).isNotNull() & (col(c) != trim(col(c)))
    ).count()
    print(f"{c}: {count}")

Distinct text values:

COLUMN: customer_name
+----------------+
|customer_name   |
+----------------+
|NULL            |
|Aarav Agarwal   |
|Aarav Chopra    |
|Aarav Iyer      |
|Aarav Kulkarni  |
|Aarav Kumar     |
|Aarav Nair      |
|Aarav Patel     |
|Aarav Singh     |
|Aarav Tripathi  |
|Abhinav Gupta   |
|Abhinav Kulkarni|
|Abhinav Malhotra|
|Abhinav Pandey  |
|Abhinav Pillai  |
|Abhinav Rao     |
|Abhinav Yadav   |
|Aditi Agarwal   |
|Aditi Chopra    |
|Aditi Kapoor    |
|Aditi Mishra    |
|Aditi Patel     |
|Aditi Tripathi  |
|Aditya Chopra   |
|Aditya Malhotra |
|Aditya Mehta    |
|Aditya Naidu    |
|Aditya Rao      |
|Akash Jain      |
|Akash Jain      |
|Akash Kulkarni  |
|Akash Pillai    |
|Akash Reddy     |
|Akash Saxena    |
|Akash Singh     |
|Akash Yadav     |
|Aman Bose       |
|Aman Chaudhary  |
|Aman Chopra     |
|Aman Das        |
|Aman Jain       |
|Aman Joshi      |
|Aman Kapoor     |
|Aman Naidu      |
|Aman Nair       |
|Aman Rao        |
|Aman Saxena     |
|Aman

In [62]:


print("QUANTITY")
print("=" * 60)
df.select("quantity").summary().show()
print("Clearly invalid quantity values (NULL or <= 0):")
df.filter(
    col("quantity").isNull() | (col("quantity") <= 0)
).select("quantity").distinct().show()

print("\nUNIT PRICE")
print("=" * 60)
df.select("unit_price").summary().show()
print("Clearly invalid unit price values (NULL or < 0):")
df.filter(
    col("unit_price").isNull() | (col("unit_price") < 0)
).select("unit_price").distinct().show()

print("\nDISCOUNT PERCENT")
print("=" * 60)
df.select("discount_percent").summary().show()
print("Clearly invalid discount values (NULL, < 0, or > 100):")
df.filter(
    col("discount_percent").isNull() |
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).select("discount_percent").distinct().show()

QUANTITY
+-------+------------------+
|summary|          quantity|
+-------+------------------+
|  count|              1000|
|   mean|             5.355|
| stddev|2.9916425128890194|
|    min|                -2|
|    25%|                 3|
|    50%|                 5|
|    75%|                 8|
|    max|                10|
+-------+------------------+

Clearly invalid quantity values (NULL or <= 0):
+--------+
|quantity|
+--------+
|      -1|
|      -2|
|       0|
+--------+


UNIT PRICE
+-------+------------------+
|summary|        unit_price|
+-------+------------------+
|  count|              1000|
|   mean|            5702.4|
| stddev|10990.194484849431|
|    min|              -500|
|    25%|               730|
|    50%|              1790|
|    75%|              4870|
|    max|             81010|
+-------+------------------+

Clearly invalid unit price values (NULL or < 0):
+----------+
|unit_price|
+----------+
|      -250|
|      -500|
+----------+


DISCOUNT PERCENT
+-------+

In [63]:

from pyspark.sql.functions import expr

date_test = df.withColumn(
    "parsed_date",
    coalesce(
        expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
        expr("try_to_timestamp(order_date, 'dd/MM/yyyy')"),
        expr("try_to_timestamp(order_date, 'MM/dd/yyyy')"),
        expr("try_to_timestamp(order_date, 'dd-MM-yyyy')"),
        expr("try_to_timestamp(order_date, 'MM-dd-yyyy')"),
        expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")
    ).cast("date")
)

# Count non-null dates that could not be parsed
invalid_date_count = date_test.filter(
    col("order_date").isNotNull() &
    col("parsed_date").isNull()
).count()

print("Invalid/unparseable non-null dates:", invalid_date_count)

# Display the invalid date values
print("\nInvalid date values:")
date_test.filter(
    col("order_date").isNotNull() &
    col("parsed_date").isNull()
).select("order_date").distinct().show(
    100,
    truncate=False
)

Invalid/unparseable non-null dates: 4

Invalid date values:
+----------+
|order_date|
+----------+
|31/02/2026|
|not-a-date|
|2025-02-30|
|2026-13-05|
+----------+



## 4. Basic preprocessing — Silver layer

Preprocessing choices:
1. Remove exact duplicate rows.
2. Replace missing text values with `Unknown`.
3. Trim leading/trailing whitespace.
4. Standardize obvious capitalization inconsistencies in human-readable text fields.
5. Canonicalize known payment-method spellings/capitalization.
6. Treat quantity <= 0 as invalid and replace invalid/missing values with the median valid quantity.
7. Treat negative unit prices as invalid and replace invalid/missing values with the median valid unit price.
8. Treat discounts outside 0–100% as invalid and replace invalid/missing values with the median valid discount.
9. Remove rows with missing `order_id`, because an order cannot be reliably identified without its identifier.
10. Convert valid dates to a consistent `yyyy-MM-dd` Spark date. Clearly invalid dates become NULL while the rest of the useful sales record is preserved.

In [64]:


silver_df = df.dropDuplicates()

print("Records after removing exact duplicates:", silver_df.count())

Records after removing exact duplicates: 990


In [65]:


for c in text_columns:
    silver_df = silver_df.withColumn(c, trim(col(c)))

initcap_columns = [
    "customer_name",
    "product",
    "category",
    "city",
    "state",
    "order_status"
]

for c in initcap_columns:
    silver_df = silver_df.withColumn(c, initcap(col(c)))

silver_df = silver_df.withColumn(
    "payment_method",
    when(lower(trim(col("payment_method"))) == "credit card", "Credit Card")
    .when(lower(trim(col("payment_method"))) == "debit card", "Debit Card")
    .when(lower(trim(col("payment_method"))) == "upi", "UPI")
    .when(lower(trim(col("payment_method"))) == "net banking", "Net Banking")
    .when(lower(trim(col("payment_method"))) == "cash on delivery", "Cash on Delivery")
    .otherwise(trim(col("payment_method")))
)

print("Text trimming and capitalization standardization completed.")

Text trimming and capitalization standardization completed.


In [66]:


for c in text_columns:
    silver_df = silver_df.fillna({c: "Unknown"})

print("Missing text values handled using 'Unknown'.")

Missing text values handled using 'Unknown'.


In [67]:


silver_df = silver_df.withColumn(
    "quantity",
    when(col("quantity") <= 0, None).otherwise(col("quantity"))
)

quantity_values = silver_df.approxQuantile("quantity", [0.5], 0.01)
median_quantity = quantity_values[0] if quantity_values else 1

silver_df = silver_df.fillna({"quantity": median_quantity})

print("Median quantity used for invalid/missing values:", median_quantity)

Median quantity used for invalid/missing values: 5.0


In [68]:


silver_df = silver_df.withColumn(
    "unit_price",
    when(col("unit_price") < 0, None).otherwise(col("unit_price"))
)

price_values = silver_df.approxQuantile("unit_price", [0.5], 0.01)
median_price = price_values[0] if price_values else 0

silver_df = silver_df.fillna({"unit_price": median_price})

print("Median unit price used for invalid/missing values:", median_price)

Median unit price used for invalid/missing values: 1790.0


In [69]:


silver_df = silver_df.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") < 0) |
        (col("discount_percent") > 100),
        None
    ).otherwise(col("discount_percent"))
)

discount_values = silver_df.approxQuantile("discount_percent", [0.5], 0.01)
median_discount = discount_values[0] if discount_values else 0

silver_df = silver_df.fillna({"discount_percent": median_discount})

print("Median discount used for invalid/missing values:", median_discount)

Median discount used for invalid/missing values: 20.0


In [70]:


before_order_id_cleanup = silver_df.count()

silver_df = silver_df.filter(col("order_id").isNotNull())

after_order_id_cleanup = silver_df.count()

print(
    "Records removed because order_id was missing:",
    before_order_id_cleanup - after_order_id_cleanup
)

Records removed because order_id was missing: 0


In [71]:


silver_df = silver_df.withColumn(
    "order_date",
    coalesce(
        expr("try_to_timestamp(trim(order_date), 'yyyy-MM-dd')"),
        expr("try_to_timestamp(trim(order_date), 'dd/MM/yyyy')"),
        expr("try_to_timestamp(trim(order_date), 'MM/dd/yyyy')"),
        expr("try_to_timestamp(trim(order_date), 'dd-MM-yyyy')"),
        expr("try_to_timestamp(trim(order_date), 'MM-dd-yyyy')"),
        expr("try_to_timestamp(trim(order_date), 'yyyy/MM/dd')")
    ).cast("date")
)

print("Date conversion completed.")

Date conversion completed.


## 5. Silver verification

In [72]:


final_count = silver_df.count()

print("=" * 60)
print("PREPROCESSING VERIFICATION")
print("=" * 60)
print("Records before preprocessing:", initial_count)
print("Records after preprocessing :", final_count)
print("Records removed             :", initial_count - final_count)

print("\nSilver schema:")
silver_df.printSchema()

print("\nSample Silver records:")
silver_df.show(10, truncate=False)

PREPROCESSING VERIFICATION
Records before preprocessing: 1000
Records after preprocessing : 990
Records removed             : 10

Silver schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)


Sample Silver records:
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|

In [73]:


final_null_counts = silver_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
])

print("Remaining NULL values in Silver:")
final_null_counts.show(truncate=False)

print("\nNote: invalid dates are preserved as NULL where they could not be parsed,")
print("so that useful sales records are not unnecessarily removed.")

Remaining NULL values in Silver:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |0            |0      |0       |0       |0         |0               |0             |0   |0    |4         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+


Note: invalid dates are preserved as NULL where they could not be parsed,
so that useful sales records are not unnecessarily removed.


## 6. Save the Silver layer

Spark may normally create multiple part files. `coalesce(1)` is used here so that the assignment produces one CSV data file that is easy to submit.

In [74]:


silver_output_path = "/content/data_lake/silver/preprocessed_sales"

silver_df.coalesce(1)     .write     .mode("overwrite")     .option("header", True)     .option("dateFormat", "yyyy-MM-dd")     .csv(silver_output_path)

print("Silver data saved successfully.")
print("Silver directory:", silver_output_path)
print("\nFiles created:")
for f in os.listdir(silver_output_path):
    print(f)

Silver data saved successfully.
Silver directory: /content/data_lake/silver/preprocessed_sales

Files created:
_SUCCESS
._SUCCESS.crc
part-00000-d306ba50-dfb7-4218-bc9c-da7f37574482-c000.csv
.part-00000-d306ba50-dfb7-4218-bc9c-da7f37574482-c000.csv.crc


In [75]:


silver_read = spark.read     .option("header", True)     .option("inferSchema", True)     .csv(silver_output_path)

print("Silver data successfully read back with Spark.")

print("\nSample rows:")
silver_read.show(10, truncate=False)

print("\nSilver record count:", silver_read.count())

Silver data successfully read back with Spark.

Sample rows:
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C022

## 7. Prepare the file required for GitHub submission

The class repository asks for:

`submissions/<ROLL_NUMBER>_<NAME>/silver/sales_silver.csv`

The following cell copies the single Spark-generated CSV part file to that exact filename.

In [76]:


submission_silver_dir = "/content/silver"
submission_silver_file = "/content/silver/sales_silver.csv"

os.makedirs(submission_silver_dir, exist_ok=True)

part_files = glob.glob(
    "/content/data_lake/silver/preprocessed_sales/part-*.csv"
)

if not part_files:
    raise FileNotFoundError("No Spark-generated Silver CSV part file was found.")

# Because coalesce(1) was used, exactly one CSV part file is expected.
shutil.copy2(part_files[0], submission_silver_file)

print("Created submission file:")
print(submission_silver_file)
print("File exists:", os.path.exists(submission_silver_file))
print("File size:", os.path.getsize(submission_silver_file), "bytes")

Created submission file:
/content/silver/sales_silver.csv
File exists: True
File size: 107544 bytes


In [77]:


submission_check = spark.read     .option("header", True)     .option("inferSchema", True)     .csv(submission_silver_file)

print("Final submission Silver file:")
submission_check.show(10, truncate=False)

submission_count = submission_check.count()

print("\nOriginal record count:", initial_count)
print("Final Silver record count:", final_count)
print("Read-back submission record count:", submission_count)
print("Counts match:", final_count == submission_count)

Final submission Silver file:
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Comp